In [1]:

# RQ4: How does experience level impact salary in AI jobs?


import pandas as pd
import numpy as np
import os
import shutil
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from scipy.stats import f_oneway

df = pd.read_csv("ai_job_dataset1.csv")

os.makedirs("models/RQ4", exist_ok=True)
os.makedirs("tables/RQ4", exist_ok=True)

windows_path = r"E:\ML ASSIGNMENT 2"
os.makedirs(windows_path, exist_ok=True)

data = df[['experience_level', 'years_experience', 'salary_usd']].dropna()

le = LabelEncoder()
data['experience_level_encoded'] = le.fit_transform(data['experience_level'])

mapping_table = pd.DataFrame({
    "Experience Level": le.classes_,
    "Encoded Value": range(len(le.classes_))
})

mapping_table.to_csv("tables/RQ4/experience_encoding_table.csv", index=False)


groups = [
    data[data['experience_level'] == lvl]['salary_usd']
    for lvl in data['experience_level'].unique()
]

anova_result = f_oneway(*groups)

anova_table = pd.DataFrame({
    "F-Statistic": [anova_result.statistic],
    "P-Value": [anova_result.pvalue]
})

anova_table.to_csv("tables/RQ4/anova_results.csv", index=False)

plt.figure(figsize=(10,6))
sns.boxplot(x='experience_level', y='salary_usd', data=data)

plt.title("RQ4: Salary Distribution by Experience Level")
plt.xlabel("Experience Level")
plt.ylabel("Salary (USD)")

plt.savefig("models/RQ4/rq4_boxplot_salary_by_experience.png")
plt.close()

mean_salary = data.groupby('experience_level')['salary_usd'].mean().reset_index()

plt.figure(figsize=(8,5))
sns.barplot(x='experience_level', y='salary_usd', data=mean_salary)

plt.title("RQ4: Average Salary by Experience Level")
plt.xlabel("Experience Level")
plt.ylabel("Average Salary")

plt.savefig("models/RQ4/rq4_barplot_avg_salary.png")
plt.close()

mean_salary.to_csv("tables/RQ4/avg_salary_by_experience.csv", index=False)

plt.figure(figsize=(10,6))
sns.scatterplot(
    x='years_experience',
    y='salary_usd',
    hue='experience_level',
    data=data
)

plt.title("RQ4: Years of Experience vs Salary")
plt.xlabel("Years of Experience")
plt.ylabel("Salary (USD)")

plt.savefig("models/RQ4/rq4_scatter_experience_salary.png")
plt.close()

X = data[['years_experience', 'experience_level_encoded']]
y = data['salary_usd']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)
pred = model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
r2 = r2_score(y_test, pred)

ml_results = pd.DataFrame({
    "MAE": [mae],
    "R2 Score": [r2]
})

ml_results.to_csv("tables/RQ4/ml_model_results.csv", index=False)

importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
}).sort_values(by="Importance", ascending=False)

importance.to_csv("tables/RQ4/feature_importance.csv", index=False)

plt.figure(figsize=(6,4))
sns.barplot(x='Importance', y='Feature', data=importance)

plt.title("RQ4: Feature Importance (Random Forest)")
plt.savefig("models/RQ4/rq4_feature_importance.png")
plt.close()

data.to_csv("tables/RQ4/processed_rq4_dataset.csv", index=False)

shutil.copy("tables/RQ4/processed_rq4_dataset.csv",
            windows_path + r"\RQ4_processed_dataset.csv")

=

print("RQ4 Analysis Completed Successfully!")
print("Files saved in:")
print("- models/RQ4 (figures)")
print("- tables/RQ4 (tables)")
print("- E:\\ML ASSIGNMENT 2 (dataset copy)")

RQ4 Analysis Completed Successfully!
Files saved in:
- models/RQ4 (figures)
- tables/RQ4 (tables)
- E:\ML ASSIGNMENT 2 (dataset copy)
